In [ ]:
!pip install open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.6 MB/s eta 0:00:00


Image2Embedding

In [ ]:
"""
Minimal CLIP Image Encoder
===========================
Only does one thing: encode all images in a folder -> save as .npz

Usage in Google Colab:
  1. pip install open-clip-torch Pillow
  2. Upload images to a folder (e.g. /content/kb_images/)
  3. Run this script
  4. Output: embeddings.npz where each key = filename without extension

Image naming examples:
  kb_01.jpg  ->  key "kb_01"
  kb_02.png  ->  key "kb_02"
"""

import os
import glob
import torch
import open_clip
import numpy as np
from PIL import Image
from pathlib import Path

# ============ EDIT THESE TWO PATHS ============
IMAGE_DIR = "./kb_images/"          # folder with your images
OUTPUT_FILE = "./kb_embeddings.npz" # where to save
# ===============================================

# Load model
device = "cuda" if torch.cuda.is_available() else "cpu"
model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
model = model.to(device).eval()
print(f"Model loaded, using {device}")

# Encode all images
embeddings = {}
paths = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.[jJpP][pPnN][gG]")))  # jpg, png

for path in paths:
    key = Path(path).stem
    img = preprocess(Image.open(path).convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        emb = model.encode_image(img)
        emb = emb / emb.norm(dim=-1, keepdim=True)
    embeddings[key] = emb.cpu().numpy().squeeze()
    print(f"  {key}  OK")

# Save
np.savez_compressed(OUTPUT_FILE, **embeddings)
print(f"\nDone: {len(embeddings)} images -> {OUTPUT_FILE}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Model loaded, using cpu
  c1  OK
  c1g  OK
  c2  OK
  c3  OK
  c4  OK
  c5  OK
  c6  OK
  c7  OK
  c8  OK
  c9  OK
  m1  OK
  m10  OK
  m11  OK
  m12  OK
  m2  OK
  m3  OK
  m4  OK
  m5g  OK
  m6  OK
  m7  OK
  m8  OK
  m9  OK
  no1  OK
  no2  OK
  no3  OK
  no4  OK
  o1  OK
  o2  OK
  o3  OK
  o4  OK

Done: 30 images -> ./kb_embeddings.npz
